# (IIP314W) Optimización Aplicada a Negocios
## Ayudantía 9: Repaso Integral del Curso

---

**Profesor:** Ing. Rodrigo Trigo Vilches  
**Ayudante:** Lic. Vicente Ramírez Almonacid  
**Fecha:** 6 de Mayo, 2026  
**Universidad del Desarrollo**

---

## Tabla de Contenidos

| # | Sección | Contenido |
|:---:|:---|:---|
| 0 | [Resumen del Curso: Hoja de Ruta](#sec0) | Bloques 1–4, fórmulas clave, herramientas |
| 1 | [Ejercicio 1 — Modelamiento Complejo (MIP)](#ex1) | FrioChile S.A.: cadena de frío farmacéutica |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex1-a) | Formulación matemática |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex1-b) | Implementación Gurobipy |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex1-c) | Análisis e interpretación |
| 2 | [Ejercicio 2 — LP Integral](#ex2) | Viñedos del Valle S.A.: Simplex → Dual → Sensibilidad |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex2-a) | Forma estándar, tableau inicial |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex2-b) | Iteraciones Simplex |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex2-c) | Formulación del dual, precios sombra |
| | &nbsp;&nbsp;&nbsp;[Parte (d)](#ex2-d) | Sensibilidad en $b_1$ |
| | &nbsp;&nbsp;&nbsp;[Parte (e)](#ex2-e) | Sensibilidad en $c_1$ |
| | &nbsp;&nbsp;&nbsp;[Parte (f)](#ex2-f) | Verificación scipy + Gurobipy |
| 3 | [Ejercicio 3 — Dualidad, Holguras Complementarias y KKT](#ex3) | Operadora Turística Costera: KKT ↔ Dualidad |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex3-a) | Recuperar $x^*$ desde $y^*$ |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex3-b) | Verificación KKT |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex3-c) | Sensibilidad en $b_2$ |
| | &nbsp;&nbsp;&nbsp;[Parte (d)](#ex3-d) | Verificación scipy |
| — | [Resumen de la Ayudantía](#resumen) | Conexiones entre bloques, tabla de herramientas |

<a id="sec0"></a>

---

## Sección 0 — Resumen del Curso: Hoja de Ruta

### Bloque 1 — Optimización No Lineal Clásica (Ayudantías 1–3)

| Concepto | Fórmula / Idea clave | Herramienta |
|:---|:---|:---|
| **Puntos críticos** | $\nabla f(x^*) = 0$ | Cálculo manual / `scipy.optimize.minimize` |
| **Clasificación Hessiana** | $H \succ 0$: mínimo local; $H \prec 0$: máximo; indef.: punto silla | `numpy.linalg.eig` |
| **Lagrange** (igualdad) | $\nabla f = \lambda \nabla g$, $g(x)=0$ | Eliminación manual |
| **KKT** (desigualdad) | Estacionaridad: $\nabla f - \sum \mu_i \nabla g_i = 0$; HC: $\mu_i g_i(x^*)=0$; Factibilidad primal: $g_i(x^*)\leq 0$; Factibilidad dual: $\mu_i \geq 0$ | Verificación manual |
| **Condición de Slater** | $\exists\, x$ tal que $g_i(x)<0\;\forall i$ → KKT son necesarias **y** suficientes | Verificación |
| **Descenso de gradiente** | $x_{k+1} = x_k - \alpha_k \nabla f(x_k)$ | `scipy.optimize.minimize` (método `'Nelder-Mead'`, `'SLSQP'`) |

> **Conexión con Bloque 4:** Las condiciones KKT ($\mu_i g_i(x^*)=0$) son exactamente las **condiciones de holgura complementaria** de la teoría dual en PL.

### Bloque 2 — Modelamiento LP y MIP (Ayudantías 4–5)

| Tipo de modelo | Ejemplo de contexto | Variable clave | Restricción típica |
|:---|:---|:---|:---|
| **LP de producción** | Maximizar margen con recursos limitados | $x_j \geq 0$ (continua) | $\sum a_{ij} x_j \leq b_i$ |
| **LP de transporte** | Flujo mínimo de plantas a ciudades | $x_{ij} \geq 0$ | Oferta, demanda |
| **MIP — localización** | Abrir/cerrar plantas, costos fijos | $y_i \in \{0,1\}$ | $\sum_j x_{ij} \leq u_i y_i$ (Big-M) |
| **MIP — lote mínimo** | Producir al menos $L$ unidades si se activa | $y_j \in \{0,1\}$ | $x_j \geq L\, y_j$ |

**Técnica Big-M:** Para ligar una variable continua $x$ a una binaria $y$:
$$x \leq M \cdot y \quad (\text{si } y=0 \Rightarrow x=0) \qquad x \geq L \cdot y \quad (\text{si } y=1 \Rightarrow x\geq L)$$

Herramientas: `scipy.optimize.linprog` (LP puro), `gurobipy` (MIP y LP de mayor escala).

---

### Bloque 3 — Algoritmo Simplex (Ayudantías 6–7)

| Concepto | Forma Tableau | Forma Matricial |
|:---|:---|:---|
| **Forma estándar** | Agregar $s_i$ (holgura, $\leq$) o $-e_i+a_i$ (excedente+artificial, $\geq$) | $[A\mid I]\begin{bmatrix}x\\s\end{bmatrix}=b$ |
| **SBF / Base** | $m$ variables básicas, resto $=0$ | $x_B = B^{-1}b\geq 0$ |
| **Costos reducidos** | Fila CR del tableau | $\bar{c}_j = c_j - c_B^\top B^{-1} a_j$ |
| **Entra** | $\min\{\bar{c}_j\}$ (más negativo) | Mismo criterio |
| **Sale** | $\min\{b_i/a_{ik}\mid a_{ik}>0\}$ | Mismo criterio |
| **Gran M** | Penalizar $a_i$ con $-M$ (max) o $+M$ (min) | Misma lógica en $c_B$ |
| **Simplex Dual** | Sale el RHS más negativo; entra $\max(|\bar{c}_j/a_{rj}|)$ para $a_{rj}<0$ | Preserva dual-factibilidad |

---

### Bloque 4 — Dualidad y Sensibilidad (Ayudantía 8)

| Concepto | Fórmula clave | Interpretación |
|:---|:---|:---|
| **Par Primal–Dual (MAX/MIN)** | Primal MAX $c^\top x$, $Ax\leq b$ $\Leftrightarrow$ Dual MIN $b^\top y$, $A^\top y\geq c$ | Variables duales = precios sombra |
| **Dualidad débil** | $b^\top y \leq c^\top x$ para cualquier par factible | El dual da cotas al primal |
| **Dualidad fuerte** | $z^* = w^*$ en el óptimo | Verificación de optimalidad |
| **Holgura complementaria** | $y_i^*(a_i^\top x^* - b_i)=0$ y $x_j^*(c_j - a_j^\top y^*)=0$ | Recuperar primal desde dual |
| **Precio sombra** | $y_i^* = \partial z^*/\partial b_i$ | Valor marginal de relajar restricción $i$ |
| **Sensibilidad en $b$** | $\bar{b} + \Delta d_i \geq 0$ donde $d_i=$ col $i$ de $B^{-1}$ | Rango donde la base no cambia |
| **Sensibilidad en $c$** | $\bar{c}_j^{\text{nuevo}}\geq 0\;\forall j\notin B$ | Rango donde la optimalidad no cambia |

**Conexión global:** KKT (Bloque 1) $\equiv$ Holguras Complementarias (Bloque 4). Los multiplicadores de Lagrange $\mu_i$ son exactamente las variables duales $y_i^*$.

<a id="ex1"></a>

---

## Ejercicio 1 — Modelamiento Complejo (MIP)

### Contexto de Negocio: Red de Distribución Refrigerada FrioChile S.A.

**FrioChile S.A.** opera una red de cadena de frío para distribuir productos farmacéuticos en dos regiones del país: Zona Norte (N) y Zona Sur (S). La empresa evalúa qué centros de distribución refrigerados (CDR) abrir durante los próximos **dos períodos** (período 1 = invierno, período 2 = verano), considerando que la demanda varía estacionalmente.

Existen **tres ubicaciones candidatas** para instalar CDRs: $k \in \{1, 2, 3\}$. Cada CDR tiene un **costo fijo de apertura** $f_k$ (en millones de pesos) que se paga solo si se decide abrir ese CDR. Una vez abierto, el CDR opera en ambos períodos. Además, cada CDR tiene una **capacidad máxima de despacho por período** $u_k$ (en toneladas).

La empresa despacha desde los CDRs abiertos hacia dos zonas $z \in \{N, S\}$ en cada período $t \in \{1, 2\}$. El **costo variable de despacho** por tonelada desde el CDR $k$ hacia la zona $z$ en el período $t$ es $c_{kzt}$. Cada zona tiene una **demanda mínima** $d_{zt}$ que debe ser satisfecha completamente.

**Restricciones adicionales:**
1. Por regulación sanitaria, **al menos 2 CDRs deben estar abiertos** (garantía de redundancia).
2. Si el CDR 1 está abierto, entonces el CDR 3 **también debe estar abierto** (el CDR 1 requiere soporte logístico del CDR 3).
3. El presupuesto total para costos fijos no puede superar **\$28 millones**.

**Parámetros numéricos:**

| CDR | Costo Fijo $f_k$ (MM\$) | Capacidad $u_k$ (ton/período) |
|:---:|:---:|:---:|
| 1 | 10 | 50 |
| 2 | 12 | 60 |
| 3 | 8 | 40 |

Costos variables $c_{kzt}$ (\$/ton):

| CDR \ Zona-Período | Zona N, t=1 | Zona S, t=1 | Zona N, t=2 | Zona S, t=2 |
|:---:|:---:|:---:|:---:|:---:|
| CDR 1 | 5 | 8 | 6 | 9 |
| CDR 2 | 7 | 4 | 8 | 5 |
| CDR 3 | 6 | 6 | 5 | 7 |

Demandas $d_{zt}$ (ton):

| Zona \ Período | $t=1$ (invierno) | $t=2$ (verano) |
|:---:|:---:|:---:|
| Zona N | 30 | 25 |
| Zona S | 20 | 35 |

<a id="ex1-a"></a>

### Parte (a) — Formulación Matemática

**Conjuntos:**
- $K = \{1, 2, 3\}$: ubicaciones candidatas de CDRs.
- $Z = \{N, S\}$: zonas de despacho.
- $T = \{1, 2\}$: períodos de planificación.

**Parámetros:**
- $f_k$: costo fijo de apertura del CDR $k$ (MM\$).
- $u_k$: capacidad máxima del CDR $k$ por período (ton).
- $c_{kzt}$: costo variable de despacho desde CDR $k$ a zona $z$ en período $t$ (\$/ton).
- $d_{zt}$: demanda de la zona $z$ en el período $t$ (ton).
- $K_{\min}$: número mínimo de CDRs que deben estar abiertos.
- $F_{\max}$: presupuesto máximo para costos fijos (MM\$).

**Variables de decisión:**
- $y_k \in \{0,1\}$: igual a 1 si se abre el CDR $k$; 0 en caso contrario.
- $x_{kzt} \geq 0$: toneladas despachadas desde CDR $k$ a zona $z$ en período $t$.

**Función objetivo** — minimizar costos totales (fijos + variables):

$$\min \quad Z = \underbrace{\sum_{k \in K} f_k \, y_k}_{\text{costos fijos}} + \underbrace{\sum_{k \in K} \sum_{z \in Z} \sum_{t \in T} c_{kzt} \, x_{kzt}}_{\text{costos variables}}$$

**Restricciones:**

**(R1) Satisfacción de demanda:**
$$\sum_{k \in K} x_{kzt} \geq d_{zt} \qquad \forall\, z \in Z,\; t \in T$$

**(R2) Capacidad condicional (Big-M):**
$$\sum_{z \in Z} x_{kzt} \leq u_k \, y_k \qquad \forall\, k \in K,\; t \in T$$

> **Nota:** si $y_k = 0$ el RHS es $0$ y se fuerza $x_{kzt} = 0$; si $y_k = 1$ se permite despachar hasta $u_k$. El parámetro $u_k$ actúa como la constante Big-M.

**(R3) Mínimo de CDRs abiertos:**
$$\sum_{k \in K} y_k \geq K_{\min}$$

**(R4) Restricción lógica CDR 1 implica CDR 3:**
$$y_1 \leq y_3$$

**(R5) Presupuesto de costos fijos:**
$$\sum_{k \in K} f_k \, y_k \leq F_{\max}$$

**(R6) Naturaleza de variables:**
$$y_k \in \{0,1\} \;\forall\, k \in K, \qquad x_{kzt} \geq 0 \;\forall\, k \in K,\, z \in Z,\, t \in T$$

<a id="ex1-b"></a>

### Parte (b) — Implementación en Gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# ─── Conjuntos ───────────────────────────────────────────────────────────────
cdrs   = [1, 2, 3]
zonas  = ['N', 'S']
periodos = [1, 2]

# ─── Parámetros ──────────────────────────────────────────────────────────────
costo_fijo = {1: 10, 2: 12, 3: 8}          # MM$
capacidad  = {1: 50, 2: 60, 3: 40}          # ton/período

# Costo variable c[k][z][t]  ($/ton)
costo_var = {
    (1, 'N', 1): 5,  (1, 'S', 1): 8,  (1, 'N', 2): 6,  (1, 'S', 2): 9,
    (2, 'N', 1): 7,  (2, 'S', 1): 4,  (2, 'N', 2): 8,  (2, 'S', 2): 5,
    (3, 'N', 1): 6,  (3, 'S', 1): 6,  (3, 'N', 2): 5,  (3, 'S', 2): 7,
}

# Demanda d[z][t]  (ton)
demanda = {
    ('N', 1): 30, ('S', 1): 20,
    ('N', 2): 25, ('S', 2): 35,
}

presupuesto_fijo = 28  # MM$

# ─── Modelo ──────────────────────────────────────────────────────────────────
modelo = gp.Model("FrioChile_ColdChain")
modelo.setParam('OutputFlag', 1)

# ─── Variables de decisión ───────────────────────────────────────────────────
# Binaria: ¿se abre el CDR k?
y = modelo.addVars(cdrs, vtype=GRB.BINARY, name="abre_cdr")

# Continua: toneladas despachadas desde CDR k a zona z en período t
x = modelo.addVars(cdrs, zonas, periodos, lb=0.0, name="despacho")

# ─── Función objetivo ────────────────────────────────────────────────────────
costo_fijo_total    = gp.quicksum(costo_fijo[k] * y[k] for k in cdrs)
costo_variable_total = gp.quicksum(
    costo_var[k, z, t] * x[k, z, t]
    for k in cdrs for z in zonas for t in periodos
)
modelo.setObjective(costo_fijo_total + costo_variable_total, GRB.MINIMIZE)

# ─── Restricciones ───────────────────────────────────────────────────────────
# R1: Satisfacción de demanda
for z in zonas:
    for t in periodos:
        modelo.addConstr(
            gp.quicksum(x[k, z, t] for k in cdrs) >= demanda[z, t],
            name=f"demanda_z{z}_t{t}"
        )

# R2: Capacidad condicional (Big-M con M = u_k)
for k in cdrs:
    for t in periodos:
        modelo.addConstr(
            gp.quicksum(x[k, z, t] for z in zonas) <= capacidad[k] * y[k],
            name=f"capacidad_cdr{k}_t{t}"
        )

# R3: Al menos 2 CDRs abiertos
modelo.addConstr(gp.quicksum(y[k] for k in cdrs) >= 2, name="min_cdrs")

# R4: CDR 1 implica CDR 3 (y1 <= y3)
modelo.addConstr(y[1] <= y[3], name="logica_cdr1_cdr3")

# R5: Presupuesto de costos fijos
modelo.addConstr(
    gp.quicksum(costo_fijo[k] * y[k] for k in cdrs) <= presupuesto_fijo,
    name="presupuesto_fijo"
)

# ─── Resolver ────────────────────────────────────────────────────────────────
modelo.optimize()

# ─── Resultados ──────────────────────────────────────────────────────────────
if modelo.status == GRB.OPTIMAL:
    print("\n" + "="*55)
    print(f"  SOLUCIÓN ÓPTIMA — Costo Total: {modelo.ObjVal:.2f} MM$")
    print("="*55)

    print("\nCDRs abiertos:")
    for k in cdrs:
        estado = "ABIERTO" if y[k].X > 0.5 else "cerrado"
        print(f"  CDR {k}: {estado}  (costo fijo = {costo_fijo[k]} MM$)")

    print("\nPlan de despacho (ton):")
    filas = []
    for k in cdrs:
        for z in zonas:
            for t in periodos:
                val = x[k, z, t].X
                if val > 1e-6:
                    filas.append({'CDR': k, 'Zona': z, 'Período': t, 'Despacho (ton)': round(val, 2)})
    df = pd.DataFrame(filas)
    print(df.to_string(index=False))

    costo_f = sum(costo_fijo[k] * y[k].X for k in cdrs)
    costo_v = sum(costo_var[k, z, t] * x[k, z, t].X
                  for k in cdrs for z in zonas for t in periodos)
    print(f"\nDesglose de costos:")
    print(f"  Costos fijos:    {costo_f:.2f} MM$")
    print(f"  Costos variables:{costo_v:.2f} MM$  (en $/ton escalados)")
    print(f"  Total:           {modelo.ObjVal:.2f} MM$")

<a id="ex1-c"></a>

### Parte (c) — Análisis e Interpretación

Una vez obtenida la solución óptima, responda las siguientes preguntas.

**1. ¿Qué CDRs se abren en la solución óptima y por qué es consistente con las restricciones lógicas?**

> **Solución pauta:** La solución óptima abre los CDRs **2 y 3** con un costo fijo total de $12 + 8 = 20$ MM\$ (dentro del presupuesto de 28 MM\$). El CDR 1 permanece cerrado. Esto es consistente con R4 ($y_1 \leq y_3$): como $y_1 = 0$, la restricción se satisface trivialmente. También se satisface R3 ($y_2 + y_3 = 2 \geq 2$). El CDR 1 no se abre porque su costo fijo (10 MM\$) no compensa frente a sus costos variables competitivos con los CDRs 2 y 3 disponibles.

**2. ¿Qué restricción R2 (capacidad condicional) está activa en el óptimo y qué implicancia tiene?**

> **Solución pauta:** Verificar en el output del modelo si algún CDR opera a capacidad plena en algún período. Si la restricción de capacidad del CDR 3 en el período 2 es activa (el CDR 3 despacha exactamente 40 ton en el período 2), indica que ampliar la capacidad del CDR 3 mejoraría el costo total. Esto corresponde a la noción de precio sombra: el valor dual de esa restricción de capacidad indica cuánto mejoraría el costo por cada tonelada adicional de capacidad.

**3. Si se relajara la restricción lógica R4 (CDR 1 ya no requiere CDR 3), ¿podría mejorar el costo total?**

> **Solución pauta:** Al relajar R4, el solver podría explorar la combinación {CDR 1, CDR 2} con costo fijo $10 + 12 = 22$ MM\$. Sin embargo, el CDR 1 tiene costos variables relativamente altos hacia la Zona S ($8$ y $9$ \$/ton). Dado que la Zona S tiene demanda importante (20 + 35 = 55 ton en total), esta combinación podría ser más cara en costos variables que la solución actual con CDR 2 (bajo costo variable en Zona S) y CDR 3. Por lo tanto, relajar R4 no garantiza mejoría significativa, pero puede ser verificado directamente en Gurobi eliminando esa restricción.

<a id="ex2"></a>

---

## Ejercicio 2 — LP Integral: Simplex Tableau, Dual y Análisis de Sensibilidad

### Contexto de Negocio: Viñedos del Valle S.A.

**Viñedos del Valle S.A.** es una empresa vitivinícola que elabora dos tipos de vino: **Reserva** ($x_1$, en miles de botellas) y **Gran Reserva** ($x_2$, en miles de botellas). El proceso productivo utiliza tres recursos compartidos:

| Recurso | Reserva ($x_1$) | Gran Reserva ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Uvas seleccionadas (toneladas) | $1$ | $1$ | $\leq 6$ ton |
| Horas de barrica (hrs) | $1$ | $2$ | $\leq 10$ hrs |
| Etiquetado artesanal (hrs) | $1$ | $0$ | $\leq 4$ hrs |
| **Margen neto (M\$/miles bot.)** | **\$2** | **\$3** | — |

La empresa desea **maximizar el margen neto** semanal.

### Formulación

$$\max \quad z = 2x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases}
x_1 + x_2 \leq 6 & \text{(uvas)} \\
x_1 + 2x_2 \leq 10 & \text{(barrica)} \\
x_1 \leq 4 & \text{(etiquetado)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

**Forma estándar** (variables de holgura $s_1, s_2, s_3 \geq 0$):

$$\max \quad z = 2x_1 + 3x_2$$
$$\text{s.a.} \quad \begin{cases}
x_1 + x_2 + s_1 = 6 \\
x_1 + 2x_2 + s_2 = 10 \\
x_1 + s_3 = 4 \\
x_1, x_2, s_1, s_2, s_3 \geq 0
\end{cases}$$

<a id="ex2-a"></a>

### Parte (a) — Forma Estándar y Tableau Inicial

El sistema extendido $[A \mid I]$ con columnas para $x_1, x_2, s_1, s_2, s_3$ es:

$$[A \mid I] = \begin{bmatrix} 1 & 1 & 1 & 0 & 0 \\ 1 & 2 & 0 & 1 & 0 \\ 1 & 0 & 0 & 0 & 1 \end{bmatrix}$$

**Base inicial:** $\{s_1, s_2, s_3\}$ — vértice $(0, 0)$, $z = 0$.

**Tableau Inicial (Iteración 0):**

La fila CR muestra los costos reducidos con signo negativo (para maximización): $\text{CR}_j = -c_j + c_B^\top B^{-1} a_j$. Con $c_B = [0,0,0]$ y $B=I$:

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $-2$ | $-3$ | $0$ | $0$ | $0$ | $0$ |
| $s_1$ | $1$ | $1$ | $1$ | $0$ | $0$ | $6$ |
| $s_2$ | $1$ | $2$ | $0$ | $1$ | $0$ | $10$ |
| $s_3$ | $1$ | $0$ | $0$ | $0$ | $1$ | $4$ |

> El costo reducido más negativo es $-3$ (columna $x_2$). **$x_2$ entra a la base.**
>
> Prueba del cociente: $6/1 = 6$, $10/2 = 5$, $4/0 = \infty$. Mínimo = $5$ (fila $s_2$). **$s_2$ sale.** Pivote: $\boxed{2}$.

<a id="ex2-b"></a>

### Parte (b) — Iteración 1

**Pivote:** elemento $a_{22} = 2$ (fila $s_2$, columna $x_2$).

**Operaciones de fila:**

$$R_{x_2}^{\text{nueva}} = \frac{R_{s_2}}{2} = \left[\frac{1}{2},\; 1,\; 0,\; \frac{1}{2},\; 0 \;\Big|\; 5\right]$$

$$R_{\text{CR}}^{\text{nueva}} = R_{\text{CR}} + 3 \cdot R_{x_2}^{\text{nueva}} = \left[-\frac{1}{2},\; 0,\; 0,\; \frac{3}{2},\; 0 \;\Big|\; 15\right]$$

$$R_{s_1}^{\text{nueva}} = R_{s_1} - 1 \cdot R_{x_2}^{\text{nueva}} = \left[\frac{1}{2},\; 0,\; 1,\; -\frac{1}{2},\; 0 \;\Big|\; 1\right]$$

$$R_{s_3}^{\text{nueva}} = R_{s_3} \quad \text{(sin cambio)}$$

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $-1/2$ | $0$ | $0$ | $3/2$ | $0$ | $15$ |
| $s_1$ | $1/2$ | $0$ | $1$ | $-1/2$ | $0$ | $1$ |
| $x_2$ | $1/2$ | $1$ | $0$ | $1/2$ | $0$ | $5$ |
| $s_3$ | $1$ | $0$ | $0$ | $0$ | $1$ | $4$ |

> CR más negativo: $-1/2$ (col. $x_1$). **$x_1$ entra.** Cocientes: $1/(1/2)=2$, $5/(1/2)=10$, $4/1=4$. Mínimo = $2$ (fila $s_1$). **$s_1$ sale.** Pivote: $\boxed{1/2}$.

#### Iteración 2 — Solución Óptima

$$R_{x_1}^{\text{nueva}} = 2 \cdot R_{s_1} = \left[1,\; 0,\; 2,\; -1,\; 0 \;\Big|\; 2\right]$$

$$R_{\text{CR}}^{\text{nueva}} = R_{\text{CR}} + \tfrac{1}{2} R_{x_1}^{\text{nueva}} = \left[0,\; 0,\; 1,\; 1,\; 0 \;\Big|\; 16\right]$$

$$R_{x_2}^{\text{nueva}} = R_{x_2} - \tfrac{1}{2} R_{x_1}^{\text{nueva}} = \left[0,\; 1,\; -1,\; 1,\; 0 \;\Big|\; 4\right]$$

$$R_{s_3}^{\text{nueva}} = R_{s_3} - 1 \cdot R_{x_1}^{\text{nueva}} = \left[0,\; 0,\; -2,\; 1,\; 1 \;\Big|\; 2\right]$$

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $0$ | $0$ | $\mathbf{1}$ | $\mathbf{1}$ | $0$ | $\mathbf{16}$ |
| $x_1$ | $1$ | $0$ | $2$ | $-1$ | $0$ | $2$ |
| $x_2$ | $0$ | $1$ | $-1$ | $1$ | $0$ | $4$ |
| $s_3$ | $0$ | $0$ | $-2$ | $1$ | $1$ | $2$ |

Todos los CR $\geq 0$ → **solución óptima.**

$$\boxed{x_1^* = 2,\quad x_2^* = 4,\quad z^* = 16 \text{ M\$}}$$

### Iteración 2 — Solución Óptima

**Pivote:** elemento $1/2$ (fila $s_1$, columna $x_1$).

**Operaciones de fila:**

$$R_{x_1}^{\text{nueva}} = 2 \cdot R_{s_1} = \left[1,\; 0,\; 2,\; -1,\; 0 \;\Big|\; 2\right]$$

$$R_{\text{CR}}^{\text{nueva}} = R_{\text{CR}} + \frac{1}{2} \cdot R_{x_1}^{\text{nueva}} = [-1/2,0,0,3/2,0|15] + 1/2 \cdot [1,0,2,-1,0|2] = \left[0,\; 0,\; 1,\; 1,\; 0 \;\Big|\; 16\right]$$

$$R_{x_2}^{\text{nueva}} = R_{x_2} - \frac{1}{2} \cdot R_{x_1}^{\text{nueva}} = [1/2,1,0,1/2,0|5] - 1/2 \cdot [1,0,2,-1,0|2] = \left[0,\; 1,\; -1,\; 1,\; 0 \;\Big|\; 4\right]$$

$$R_{s_3}^{\text{nueva}} = R_{s_3} - 1 \cdot R_{x_1}^{\text{nueva}} = [1,0,0,0,1|4] - [1,0,2,-1,0|2] = \left[0,\; 0,\; -2,\; 1,\; 1 \;\Big|\; 2\right]$$

**Tableau Óptimo — Iteración 2:**

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | $0$ | $0$ | $\mathbf{1}$ | $\mathbf{1}$ | $0$ | $\mathbf{16}$ |
| $x_1$ | $1$ | $0$ | $2$ | $-1$ | $0$ | $2$ |
| $x_2$ | $0$ | $1$ | $-1$ | $1$ | $0$ | $4$ |
| $s_3$ | $0$ | $0$ | $-2$ | $1$ | $1$ | $2$ |

**Condición de optimalidad:** todos los costos reducidos son $\geq 0$ (fila CR: $0, 0, 1, 1, 0$). La solución es óptima.

$$\boxed{x_1^* = 2 \text{ (miles de bot. Reserva)}, \quad x_2^* = 4 \text{ (miles de bot. Gran Reserva)}, \quad z^* = 16 \text{ M\$}}$$

Variables no básicas: $s_1 = 0$, $s_2 = 0$ (restricciones 1 y 2 **activas**). $s_3 = 2$ (restricción 3 **inactiva** — sobran 2 horas de etiquetado).

<a id="ex2-c"></a>

### Parte (c) — Formulación del Dual

El primal es de maximización con restricciones $\leq$. El dual es de **minimización** con restricciones $\geq$ y variables $y_i \geq 0$:

$$\min \quad w = 6y_1 + 10y_2 + 4y_3$$

$$\text{s.a.} \quad \begin{cases}
y_1 + y_2 + y_3 \geq 2 & \text{(restricción dual para } x_1\text{)} \\
y_1 + 2y_2 \geq 3 & \text{(restricción dual para } x_2\text{)} \\
y_1, y_2, y_3 \geq 0
\end{cases}$$

**Precios sombra del tableau óptimo** (fila CR en columnas de las holguras):

$$y_1^* = 1,\qquad y_2^* = 1,\qquad y_3^* = 0$$

**Dualidad fuerte:** $w^* = 6(1) + 10(1) + 4(0) = 16 = z^* \;\checkmark$

**Interpretación:** $y_1^* = 1$ M\$/ton (uvas) y $y_2^* = 1$ M\$/hr (barrica) son los recursos limitantes. El etiquetado ($y_3^* = 0$) tiene capacidad sobrante ($s_3 = 2$).

<a id="ex2-d"></a>

### Parte (d) — Análisis de Sensibilidad sobre $b_1$ (uvas)

Sea $b_1 \to 6 + \Delta$. Primera columna de $B^{-1}$: $d_1 = [2, -1, -2]^\top$.

$$x_B = \begin{bmatrix}2\\4\\2\end{bmatrix} + \Delta \begin{bmatrix}2\\-1\\-2\end{bmatrix} \geq 0 \quad\Rightarrow\quad \Delta \geq -1 \;\text{ y }\; \Delta \leq 1$$

$$\boxed{5 \leq b_1 \leq 7 \text{ toneladas}} \qquad (y_1^* = 1 \text{ M\$/ton, válido en este rango})$$

<a id="ex2-e"></a>

### Parte (e) — Análisis de Sensibilidad sobre $c_1$ (margen del Reserva)

$x_1$ es básica. Con $c_1 \to 2 + \Delta$:

$$\bar{c}_{s_1} = 1 + 2\Delta \geq 0 \;\Rightarrow\; \Delta \geq -\tfrac{1}{2} \qquad \bar{c}_{s_2} = 1 - \Delta \geq 0 \;\Rightarrow\; \Delta \leq 1$$

$$\boxed{\tfrac{3}{2} \leq c_1 \leq 3 \text{ M\$/miles bot.}}$$

<a id="ex2-f"></a>

### Parte (f) — Verificación Computacional

In [ ]:
import numpy as np
from scipy.optimize import linprog

# linprog minimiza: negamos la función objetivo
c_scipy = [-2, -3]

# A_ub @ x <= b_ub
A_ub = [
    [1, 1],   # uvas
    [1, 2],   # barrica
    [1, 0],   # etiquetado
]
b_ub = [6, 10, 4]

bounds = [(0, None), (0, None)]

resultado = linprog(c_scipy, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print("=" * 45)
print("Verificación con scipy.optimize.linprog")
print("=" * 45)
print(f"x1* = {resultado.x[0]:.4f}  (Reserva, miles bot.)")
print(f"x2* = {resultado.x[1]:.4f}  (Gran Reserva, miles bot.)")
print(f"z*  = {-resultado.fun:.4f} M$")
print()
print("Precios sombra (ineq_marginals / dual_variables):")
# En scipy HiGHS, los marginals para restricciones de desigualdad
marginals = resultado.ineqlin.marginals
nombres_r = ['Uvas (b1)', 'Barrica (b2)', 'Etiquetado (b3)']
for nombre, val in zip(nombres_r, marginals):
    print(f"  {nombre}: y* = {-val:.4f} M$/unidad")
print()
print("Interpretación: -marginal porque linprog minimiza (negado).")

In [ ]:
import gurobipy as gp
from gurobipy import GRB

modelo_vino = gp.Model("Vinedos_del_Valle")
modelo_vino.setParam('OutputFlag', 0)

# Variables de decisión
x1 = modelo_vino.addVar(lb=0, name="Reserva")
x2 = modelo_vino.addVar(lb=0, name="Gran_Reserva")

# Función objetivo
modelo_vino.setObjective(2*x1 + 3*x2, GRB.MAXIMIZE)

# Restricciones
r_uvas      = modelo_vino.addConstr(x1 + x2   <= 6,  name="uvas")
r_barrica   = modelo_vino.addConstr(x1 + 2*x2 <= 10, name="barrica")
r_etiquetado = modelo_vino.addConstr(x1        <= 4,  name="etiquetado")

modelo_vino.optimize()

print("=" * 45)
print("Verificación con Gurobipy")
print("=" * 45)
print(f"x1* = {x1.X:.4f}  (Reserva, miles bot.)")
print(f"x2* = {x2.X:.4f}  (Gran Reserva, miles bot.)")
print(f"z*  = {modelo_vino.ObjVal:.4f} M$")
print()
print("Precios sombra (Pi = dual variable de cada restricción):")
for constr in [r_uvas, r_barrica, r_etiquetado]:
    print(f"  {constr.ConstrName}: y* = {constr.Pi:.4f} M$/unidad")
print()
print("Rangos de sensibilidad de la función objetivo:")
for var in [x1, x2]:
    print(f"  c({var.VarName}): [{var.SAObjLow:.4f}, {var.SAObjUp:.4f}]")
print()
print("Rangos de sensibilidad del RHS:")
for constr in [r_uvas, r_barrica, r_etiquetado]:
    print(f"  b({constr.ConstrName}): [{constr.SARHSLow:.4f}, {constr.SARHSUp:.4f}]")

### Tabla Comparativa de Resultados

| Método | $x_1^*$ | $x_2^*$ | $z^*$ | $y_1^*$ | $y_2^*$ | $y_3^*$ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Simplex Tableau (manual) | $2$ | $4$ | $16$ | $1$ | $1$ | $0$ |
| `scipy.optimize.linprog` | $2$ | $4$ | $16$ | $1$ | $1$ | $0$ |
| `gurobipy` | $2$ | $4$ | $16$ | $1$ | $1$ | $0$ |

**Rangos de sensibilidad verificados:**

| Parámetro | Rango manual | Rango Gurobi |
|:---|:---:|:---:|
| $b_1$ (uvas, ton) | $[5,\; 7]$ | `SARHSLow=5, SARHSUp=7` |
| $c_1$ (margen Reserva) | $[3/2,\; 3]$ | `SAObjLow=1.5, SAObjUp=3.0` |

<a id="ex3"></a>

---

## Ejercicio 3 — Dualidad, Holguras Complementarias y KKT

### Contexto de Negocio: Operadora Turística Costera

Una **operadora turística costera** programa excursiones en dos modalidades: **Kayak** ($x_1$, en decenas de grupos por semana) y **Vela** ($x_2$, en decenas de grupos por semana). La empresa quiere **maximizar su ingreso neto** semanal:

$$\max \quad z = x_1 + 2x_2$$

| Restricción | Kayak ($x_1$) | Vela ($x_2$) | Límite |
|:---|:---:|:---:|:---:|
| Guías disponibles | $1$ | $1$ | $\leq 5$ horas |
| Embarcaciones reservadas | $1$ | $3$ | $\leq 9$ horas |

$$\text{s.a.} \quad \begin{cases} x_1 + x_2 \leq 5 \\ x_1 + 3x_2 \leq 9 \\ x_1,\, x_2 \geq 0 \end{cases}$$

Se le informa que la **solución óptima del dual** es: $\quad y_1^* = \dfrac{1}{2}, \quad y_2^* = \dfrac{1}{2}$

<a id="ex3-a"></a>

### Parte (a) — Recuperar $x^*$ mediante Holguras Complementarias

$y_1^* > 0 \Rightarrow$ restricción 1 activa: $x_1 + x_2 = 5$

$y_2^* > 0 \Rightarrow$ restricción 2 activa: $x_1 + 3x_2 = 9$

Restando: $2x_2 = 4 \Rightarrow x_2^* = 2$, $x_1^* = 3$.

$$\boxed{x_1^* = 3,\quad x_2^* = 2,\quad z^* = 7 \text{ M\$}}$$

<a id="ex3-b"></a>

### Parte (b) — Verificación de Condiciones KKT

**1. Estacionaridad** ($A^\top y^* = c$):
$$\begin{bmatrix}1&1\\1&3\end{bmatrix}\begin{bmatrix}1/2\\1/2\end{bmatrix} = \begin{bmatrix}1\\2\end{bmatrix} = c \;\checkmark$$

**2. Holgura complementaria** ($\mu_i(b_i - a_i^\top x^*)=0$):
- $\frac{1}{2}(5-5) = 0\;\checkmark$, $\frac{1}{2}(9-9)=0\;\checkmark$

**3. Factibilidad primal:** $3+2=5\leq5\;\checkmark$, $3+6=9\leq9\;\checkmark$, $x^*\geq0\;\checkmark$

**4. Factibilidad dual:** $y_1^*, y_2^* = \frac{1}{2} \geq 0\;\checkmark$

> **Conexión conceptual:** $\mu_i \equiv y_i^*$. Las condiciones KKT del Bloque 1 y las Holguras Complementarias del Bloque 4 son la misma condición vista desde distintas perspectivas.

<a id="ex3-c"></a>

### Parte (c) — Análisis de Sensibilidad sobre $b_2$ (embarcaciones)

$$B^{-1} = \begin{bmatrix}3/2 & -1/2\\-1/2 & 1/2\end{bmatrix}, \qquad d_2 = [-1/2,\; 1/2]^\top$$

Sea $b_2 \to 9 + \Delta$:

$$x_1 = 3 - \tfrac{\Delta}{2} \geq 0 \;\Rightarrow\; \Delta \leq 6 \qquad x_2 = 2 + \tfrac{\Delta}{2} \geq 0 \;\Rightarrow\; \Delta \geq -4$$

$$\boxed{5 \leq b_2 \leq 15} \qquad y_2^* = \tfrac{1}{2} \text{ M\$/hora, válido en este rango}$$

**Dualidad fuerte:** $w^* = 5\cdot\tfrac{1}{2} + 9\cdot\tfrac{1}{2} = 7 = z^*\;\checkmark$

<a id="ex3-d"></a>

### Parte (d) — Verificación con scipy

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Primal: max z = x1 + 2*x2  →  min -x1 - 2*x2
c_scipy = [-1, -2]

A_ub = [
    [1, 1],   # guías
    [1, 3],   # embarcaciones
]
b_ub = [5, 9]
bounds = [(0, None), (0, None)]

res = linprog(c_scipy, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print("=" * 50)
print("Ejercicio 3 — Verificación scipy.optimize.linprog")
print("=" * 50)
print(f"x1* = {res.x[0]:.4f}  (dec. grupos Kayak)")
print(f"x2* = {res.x[1]:.4f}  (dec. grupos Vela)")
print(f"z*  = {-res.fun:.4f} M$")
print()
print("Precios sombra (variables duales):")
y1 = -res.ineqlin.marginals[0]
y2 = -res.ineqlin.marginals[1]
print(f"  y1* (guias)        = {y1:.4f} M$/hora")
print(f"  y2* (embarcaciones)= {y2:.4f} M$/hora")
print()
print("Verificación condiciones KKT:")
A = np.array([[1,1],[1,3]])
c = np.array([1, 2])
y_opt = np.array([y1, y2])
x_opt = res.x
b_vec = np.array([5, 9])

estacionaridad = np.allclose(A.T @ y_opt, c, atol=1e-8)
holgura_comp   = np.allclose((b_vec - A @ x_opt) * y_opt, 0, atol=1e-8)
dualidad_fuerte = np.isclose(c @ x_opt, b_vec @ y_opt, atol=1e-8)

print(f"  A'y* = c  (Estacionaridad):     {estacionaridad}")
print(f"  y*(b-Ax*) = 0 (HC primal):      {holgura_comp}")
print(f"  c'x* = b'y* (Dualidad fuerte):  {dualidad_fuerte}")
print(f"  z* = {c @ x_opt:.4f}, w* = {b_vec @ y_opt:.4f}")

<a id="resumen"></a>

---

## Resumen de la Ayudantía

### Conexiones entre bloques del curso

| Bloque | Concepto central | Conexión con otro bloque |
|:---|:---|:---|
| **1 — Optimización NL** | KKT: $\mu_i g_i(x^*)=0$, $\nabla f = \sum \mu_i \nabla g_i$ | $\mu_i \equiv y_i^*$ del Bloque 4 |
| **2 — Modelamiento MIP** | Big-M: $x \leq M y$ vincula continua con binaria | Las holguras del LP base son las variables duales del Bloque 4 |
| **3 — Simplex** | $B^{-1}$ genera precios sombra en la fila CR | $c_B B^{-1} = y^{\top}$ (precios sombra = variables duales óptimas) |
| **4 — Dualidad** | $z^* = w^*$, $y_i^* = \partial z^*/\partial b_i$ | KKT (Bloque 1) y Holguras Complementarias son equivalentes |

### Tabla de herramientas computacionales

| Tarea | Herramienta | Función/Método clave |
|:---|:---|:---|
| Optimización no lineal sin restricciones | `scipy.optimize.minimize` | `method='BFGS'` |
| Optimización no lineal con restricciones | `scipy.optimize.minimize` | `method='SLSQP'` |
| LP (verificación) | `scipy.optimize.linprog` | `method='highs'`; `.ineqlin.marginals` |
| LP / MIP (industrial) | `gurobipy` | `.optimize()`, `.Pi`, `.SAObjLow/Up`, `.SARHSLow/Up` |
| Operaciones matriciales Simplex | `numpy` | `np.linalg.inv(B)`, `c_B @ B_inv` |

---

*Ayudantía 9 (Final) — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T1*